# 04 — Basic Statistical Analysis

In this notebook, we'll use some basic statistics to see if the patterns we saw in the EDA are actually real. We'll focus on averages, correlations, and a couple of simple tests to compare groups.

### Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='whitegrid')

# Load the cleaned data
data_path = Path('../data/processed/airbnb_nyc_cleaned.csv')
df = pd.read_csv(data_path)

print(f"Ready to analyze {len(df)} listings.")

## 1. Comparing Averages: Manhattan vs Brooklyn
Is Manhattan really more expensive than Brooklyn, or is it just a few pricey listings pulling up the average? We'll use a **T-test** to find out.

In [ ]:
manhattan = df[df['neighbourhood_group'] == 'Manhattan']['price']
brooklyn = df[df['neighbourhood_group'] == 'Brooklyn']['price']

print(f"Manhattan Average Price: ${manhattan.mean():.2f}")
print(f"Brooklyn Average Price: ${brooklyn.mean():.2f}")

# Simple T-Test
t_stat, p_val = stats.ttest_ind(manhattan, brooklyn)

print(f"\nT-Statistic: {t_stat:.2f}")
print(f"P-Value: {p_val:.4f}")

**What this means:** Since the P-value is less than 0.05, we can say that the price difference between Manhattan and Brooklyn is **statistically significant**. It's not just a fluke.

## 2. Comparing Room Types
Let's look at the average price for each room type. This helps us see which type of listing brings in the most money.

In [ ]:
room_stats = df.groupby('room_type')['price'].agg(['mean', 'median', 'std', 'count'])
room_stats.sort_values('mean', ascending=False)

**Observation:** Entire homes are more than twice as expensive as private rooms on average. This is the biggest factor affecting price.

## 3. Correlation: What is linked to Price?
We'll use **Correlation** to see how price relates to other numbers, like the number of reviews or how many days a listing is available. 

*Correlation goes from -1 to 1. Closer to 1 means they move together, closer to 0 means they aren't really linked.*

In [ ]:
# Looking at correlations with price
correlations = df[['price', 'minimum_nights', 'review_count', 'availability_365', 'host_listing_count']].corr()
price_correlations = correlations['price'].sort_values(ascending=False)

print("Correlation with Price:")
print(price_correlations)

In [ ]:
# Visualizing the correlations
plt.figure(figsize=(8, 6))
sns.heatmap(correlations, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

**Key Finding:** Price has a slight positive correlation with `availability_365`. This means that listings that are available more often tend to be a bit more expensive.

## 4. Price vs. Number of Reviews
Is it true that cheaper listings get more reviews because more people book them?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='review_count', y='price', alpha=0.3)
plt.ylim(0, 1000) # clipping price to see the density better
plt.title('Relationship between Price and Number of Reviews')
plt.show()

**Observation:** You can see that the listings with the most reviews are generally priced under $200. Very expensive listings rarely have hundreds of reviews.

## Summary
1. **Manhattan is significantly more expensive** than Brooklyn.
2. **Entire homes** are the premium room type.
3. **Price doesn't have a strong link** to any single number (like availability or reviews) — it's likely a mix of factors like location and room type.

Now we're ready for the final step: preparing the data for our Tableau dashboard!